In [1]:
import gradio as gr
import json
from dotenv import load_dotenv
from openai import OpenAI
import os
from agents.classifying_agent import ClassifyingAgent
import chromadb
from agents.agent import Agent

In [2]:
DB = "products_vectorstore"
client = chromadb.PersistentClient(path=DB)
collections = client.get_or_create_collection('products')

In [ ]:
classifyingAgent_agent = ClassifyingAgent(collection=collections)


Using device: mps


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
categorization_agent = {
    "name": "categorization_agent",
    "description": "Categorizes a product based on its description. If the product is difficult to categorize, the agent will return the 3 categories you think it could belong to.",
    "parameters": {
        "type": "object",
        "properties": {
            "product_description": {
                "type": "string",
                "description": "A description of the product that needs to be categorized."
            },
            "product_attributes": {
                "type": "object",
                "description": "A dictionary of product attributes and their values."
            }
        },
        "required": ["product_description"]
    }
}




price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

In [5]:
tools = [{type: "function", "function": categorization_agent}]

In [6]:

load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
MODEL = "gpt-4.1-mini"
openai = OpenAI()

OpenAI API Key exists and begins sk-proj-


In [7]:
system_message = """
You are a helpful assistant for an e-commerce platform that helps bussinesses register their products. 
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

In [8]:
def handle_tool_call(self, message):
    """
    Actually call the tools associated with this message
    """
    mapping = {
        # "scan_the_internet_for_bargains": self.scan_the_internet_for_bargains,
        "categorization_agent": classifyingAgent_agent.classify,
        # "notify_function": fetch_manager,
    }
    results = []
    for tool_call in message.tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        tool = mapping.get(tool_name)
        result = tool(**arguments) if tool else ""
        results.append({"role": "tool", "content": result, "tool_call_id": tool_call.id})
    return results

In [9]:
WELCOME_MESSAGE = "Hi! My name is Peeta, and I'm here to help you register your product."

def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]

    response = openai.chat.completions.create(
        model=MODEL,
        messages=messages, tools=tools
    )

    # if response.choices[0].finish_reason=="tool_calls":
    #     message = response.choices[0].message
    #     response = handle_tool_call(message)
    #     messages.append(message)
    #     messages.append(response)
    #     response = openai.chat.completions.create(model=MODEL, messages=messages)

    return response.choices[0].message.content


chatbot = gr.Chatbot(
    type="messages",
    value=[
        {"role": "assistant", "content": WELCOME_MESSAGE}
    ]
)



/var/folders/_t/4vpfft894ddd5ymy303fdf2m0000gn/T/ipykernel_7900/3649053071.py:22: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(


In [10]:
gr.ChatInterface(
    fn=chat,
    chatbot=chatbot,
    type="messages"
).launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "/Users/arumlee/projectLLMAgent/LLM_AgenticAI/.venv/lib/python3.12/site-packages/gradio/queueing.py", line 759, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/arumlee/projectLLMAgent/LLM_AgenticAI/.venv/lib/python3.12/site-packages/gradio/route_utils.py", line 354, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/arumlee/projectLLMAgent/LLM_AgenticAI/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 2191, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/arumlee/projectLLMAgent/LLM_AgenticAI/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 1696, in call_function
    prediction = await fn(*processed_input)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/arumlee/projectLLMAgent/LLM_AgenticAI

In [12]:
def chat2(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "/Users/arumlee/projectLLMAgent/LLM_AgenticAI/.venv/lib/python3.12/site-packages/gradio/queueing.py", line 759, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/arumlee/projectLLMAgent/LLM_AgenticAI/.venv/lib/python3.12/site-packages/gradio/route_utils.py", line 354, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/arumlee/projectLLMAgent/LLM_AgenticAI/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 2191, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/arumlee/projectLLMAgent/LLM_AgenticAI/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 1696, in call_function
    prediction = await fn(*processed_input)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/arumlee/projectLLMAgent/LLM_AgenticAI